## 1. Hugging Face Login

#### This imports the token from Kaggle Secrets, so make sure the secret is set correctly before running the code.

In [1]:
import os
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

print("Hugging Face login successful.")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Hugging Face login successful.


## 2. Check GPU

In [2]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))

CUDA available: True
GPU count: 2
GPU 0: Tesla T4
GPU 1: Tesla T4


## 3. Configuration

#### this is sample configuration, you are free to modify any parameter to train the best performing model

In [3]:
import os
import random
import json
import time
import math
import zipfile
import shutil

# Hugging Face dataset
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"

# Dataset files
TRAIN_FILE = "Phase_1_Pretraining/Train/train_supreme_court.json"
TEST_FILE = "Phase_1_Pretraining/Test/test.json"

# ------------------------------------------------------------
# Training configuration
# ------------------------------------------------------------

TRAIN_TOKENS = 150_000_000
VAL_TOKENS = 5_000_000
TRAIN_BLOCK_SIZE = 2048

EPOCHS = 1
SEED = 42
random.seed(SEED)

MODEL_CONTEXT_LENGTH = 2048

# ------------------------------------------------------------
# CUSTOM DECODER TRANSFORMER ARCHITECTURE
# ------------------------------------------------------------

N_EMBD = 640
N_LAYER = 8
N_HEAD = 10
N_FF = 2560

# 640 / 10 = 64 dimensions per attention head.
assert N_EMBD % N_HEAD == 0

# ------------------------------------------------------------
# Generation
# ------------------------------------------------------------

TARGET_GENERATION_WORDS = 128
MAX_GENERATION_TOKENS = 500

TEMPERATURE = 0.4
TOP_P = 0.90

# ------------------------------------------------------------
# Kaggle output
# ------------------------------------------------------------

OUTPUT_DIR = (
    "/kaggle/working/"
    "cs781_phase1_a1_decodertransformer_8L10H"
)

MODEL_OUTPUT_DIR = os.path.join(
    OUTPUT_DIR,
    "custom-decoder-final"
)

PREDICTION_PATH = os.path.join(
    OUTPUT_DIR,
    "prediction.json"
)

SUBMISSION_ZIP = os.path.join(
    OUTPUT_DIR,
    "submission.zip"
)

LOSS_PLOT_PATH = os.path.join(
    OUTPUT_DIR,
    "training_validation_loss.png"
)

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

print("Configuration loaded.")
print(f"Embedding dimension: {N_EMBD}")
print(f"Layers: {N_LAYER}")
print(f"Heads: {N_HEAD}")
print(f"Head dimension: {N_EMBD // N_HEAD}")
print(f"FFN dimension: {N_FF}")

Configuration loaded.
Embedding dimension: 640
Layers: 8
Heads: 10
Head dimension: 64
FFN dimension: 2560


## 4. Load Tokenizer

In [4]:
from transformers import PreTrainedTokenizerFast
from pathlib import Path

# ------------------------------------------------------------
# Exact path of the saved tokenizer notebook output
# ------------------------------------------------------------

DATA_DIR = Path(
    "/kaggle/input/notebooks/pranavbhosle1/bpetokenizer/phase1_simple_data"
)

TOKENIZER_PATH = DATA_DIR / "tokenizer.json"
TRAIN_TOKEN_FILE = DATA_DIR / "train_tokens.bin"
VAL_TOKEN_FILE = DATA_DIR / "val_tokens.bin"


# ------------------------------------------------------------
# Load your saved 50k BPE tokenizer
# ------------------------------------------------------------

tokenizer = PreTrainedTokenizerFast(
    tokenizer_file=str(TOKENIZER_PATH),
    pad_token="<pad>",
    unk_token="<unk>",
    bos_token="<s>",
    eos_token="</s>"
)


print("Tokenizer/data folder:")
print(DATA_DIR)

print("\nVocabulary size:")
print(len(tokenizer))

print("\nTraining token file:")
print(TRAIN_TOKEN_FILE)

print("\nValidation token file:")
print(VAL_TOKEN_FILE)

Tokenizer/data folder:
/kaggle/input/notebooks/pranavbhosle1/bpetokenizer/phase1_simple_data

Vocabulary size:
50000

Training token file:
/kaggle/input/notebooks/pranavbhosle1/bpetokenizer/phase1_simple_data/train_tokens.bin

Validation token file:
/kaggle/input/notebooks/pranavbhosle1/bpetokenizer/phase1_simple_data/val_tokens.bin


In [5]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    if "tokenizer.json" in files:
        print("FOUND:", root)
        print(files)

FOUND: /kaggle/input/notebooks/pranavbhosle1/bpetokenizer/phase1_simple_data
['train_tokens.bin', 'bpe_training_text.txt', 'tokenizer.json', 'val_tokens.bin']


## 5. Download Training Data From Hugging Face

In [6]:

# ------------------------------------------------------------
# The tokenizer notebook has ALREADY prepared exactly:
#   200M training token IDs
#   5M validation token IDs
#
# Therefore we do not download and tokenize the raw training JSON again.
# We directly reuse the saved .bin files.
# ------------------------------------------------------------

TRAIN_TOKEN_FILE = os.path.join(DATA_DIR, "train_tokens.bin")
VAL_TOKEN_FILE = os.path.join(DATA_DIR, "val_tokens.bin")

print("Training token file:")
print(TRAIN_TOKEN_FILE)

print("Validation token file:")
print(VAL_TOKEN_FILE)


Training token file:
/kaggle/input/notebooks/pranavbhosle1/bpetokenizer/phase1_simple_data/train_tokens.bin
Validation token file:
/kaggle/input/notebooks/pranavbhosle1/bpetokenizer/phase1_simple_data/val_tokens.bin


## 6. Create Train and Validation Tokens Set

In [7]:

import numpy as np

# ------------------------------------------------------------
# Memory-map the token files.
#
# np.memmap is useful here because 200M token IDs are large.
# It reads small pieces from disk only when they are needed,
# instead of loading the whole file into RAM.
#
# uint16 is enough because our vocabulary is about 50k,
# and 50,000 < 65,536.
# ------------------------------------------------------------

train_tokens = np.memmap(
    TRAIN_TOKEN_FILE,
    dtype=np.uint16,
    mode="r",
    shape=(TRAIN_TOKENS,)
)

val_tokens = np.memmap(
    VAL_TOKEN_FILE,
    dtype=np.uint16,
    mode="r",
    shape=(VAL_TOKENS,)
)

print(f"Train tokens: {len(train_tokens):,}")
print(f"Validation tokens: {len(val_tokens):,}")


Train tokens: 150,000,000
Validation tokens: 5,000,000


## 7. Create Token Blocks

In [8]:

from torch.utils.data import Dataset

class TokenBlockDataset(Dataset):
    """
    Read one fixed-size block directly from the long token file.

    Example with block_size = 4:
        token file: [10, 20, 30, 40, 50, 60, 70, 80]

        block 0 -> [10, 20, 30, 40]
        block 1 -> [50, 60, 70, 80]

    We do not create a huge Python list of all blocks.
    """

    def __init__(self, tokens, block_size):
        self.tokens = tokens
        self.block_size = block_size
        self.num_blocks = len(tokens) // block_size

    def __len__(self):
        return self.num_blocks

    def __getitem__(self, index):
        start = index * self.block_size
        end = start + self.block_size

        # Convert only this block to a normal integer tensor.
        block = np.array(
            self.tokens[start:end],
            dtype=np.int64
        )

        return {
            "input_ids": torch.tensor(
                block,
                dtype=torch.long
            )
        }

print("Token block dataset class created.")


Token block dataset class created.


## 8. Create Datasets

In [9]:

train_dataset = TokenBlockDataset(
    train_tokens,
    TRAIN_BLOCK_SIZE
)

val_dataset = TokenBlockDataset(
    val_tokens,
    TRAIN_BLOCK_SIZE
)

print("Training sequences:", len(train_dataset))
print("Validation sequences:", len(val_dataset))


Training sequences: 73242
Validation sequences: 2441


## 9. Define The Model

In [10]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers.modeling_outputs import CausalLMOutput


# ============================================================
# ARCHITECTURE 1:
# CUSTOM DECODER-ONLY TRANSFORMER FROM SCRATCH
# ============================================================


class DecoderBlock(nn.Module):
    """
    One Transformer decoder block.

    Structure:
        LayerNorm
        -> Causal Multi-Head Self Attention
        -> Residual Connection
        -> LayerNorm
        -> Feed Forward Network
        -> Residual Connection
    """

    def __init__(
        self,
        d_model,
        n_heads,
        d_ff,
        dropout=0.1
    ):
        super().__init__()

        # Normalize before self-attention.
        self.norm1 = nn.LayerNorm(
            d_model
        )

        # Multi-head self-attention.
        #
        # Query, Key and Value all come from
        # the same token sequence.
        self.attention = nn.MultiheadAttention(
            embed_dim=d_model,
            num_heads=n_heads,
            dropout=dropout,
            batch_first=True
        )

        # Normalize before feed-forward network.
        self.norm2 = nn.LayerNorm(
            d_model
        )

        # Transformer feed-forward network.
        self.feed_forward = nn.Sequential(
            nn.Linear(
                d_model,
                d_ff
            ),
            nn.GELU(),
            nn.Linear(
                d_ff,
                d_model
            )
        )

        self.dropout = nn.Dropout(
            dropout
        )


    def forward(
        self,
        x,
        causal_mask
    ):

        # ----------------------------------------------------
        # 1. Causal self-attention
        # ----------------------------------------------------

        z = self.norm1(x)

        attention_output, _ = self.attention(
            z,
            z,
            z,
            attn_mask=causal_mask,
            need_weights=False
        )

        # Residual connection.
        x = x + self.dropout(
            attention_output
        )

        # ----------------------------------------------------
        # 2. Feed-forward network
        # ----------------------------------------------------

        z = self.norm2(x)

        ff_output = self.feed_forward(
            z
        )

        # Second residual connection.
        x = x + self.dropout(
            ff_output
        )

        return x



class DecoderTransformerLM(nn.Module):
    """
    Complete custom decoder-only Transformer
    language model.
    """

    def __init__(
        self,
        vocab_size,
        context_length=2048,
        d_model=640,
        n_heads=10,
        n_layers=8,
        d_ff=2560,
        dropout=0.1
    ):
        super().__init__()

        self.context_length = (
            context_length
        )

        self.vocab_size = (
            vocab_size
        )

        # Safety check:
        # embedding dimension must divide evenly
        # across attention heads.
        assert (
            d_model % n_heads == 0
        ), (
            "d_model must be divisible "
            "by n_heads"
        )

        # ----------------------------------------------------
        # Token embeddings
        # ----------------------------------------------------

        self.token_embedding = nn.Embedding(
            vocab_size,
            d_model
        )

        # ----------------------------------------------------
        # Learned positional embeddings
        # ----------------------------------------------------

        self.position_embedding = nn.Embedding(
            context_length,
            d_model
        )

        # ----------------------------------------------------
        # Stack of custom Transformer decoder blocks
        # ----------------------------------------------------

        self.blocks = nn.ModuleList([
            DecoderBlock(
                d_model=d_model,
                n_heads=n_heads,
                d_ff=d_ff,
                dropout=dropout
            )
            for _ in range(n_layers)
        ])

        # Final normalization.
        self.final_norm = nn.LayerNorm(
            d_model
        )

        # Convert hidden representation
        # into vocabulary logits.
        self.output_layer = nn.Linear(
            d_model,
            vocab_size,
            bias=False
        )


    def forward(
        self,
        input_ids,
        labels=None,
        attention_mask=None,
        **kwargs
    ):

        _, seq_len = input_ids.shape

        if seq_len > self.context_length:
            raise ValueError(
                f"Sequence length {seq_len} exceeds "
                f"context length {self.context_length}"
            )

        # ----------------------------------------------------
        # 1. Token embeddings
        # ----------------------------------------------------

        token_vectors = (
            self.token_embedding(
                input_ids
            )
        )

        # ----------------------------------------------------
        # 2. Positional embeddings
        # ----------------------------------------------------

        positions = torch.arange(
            seq_len,
            device=input_ids.device
        )

        position_vectors = (
            self.position_embedding(
                positions
            )
        )

        x = (
            token_vectors
            +
            position_vectors
        )

        # ----------------------------------------------------
        # 3. Causal mask
        #
        # True = attention blocked.
        #
        # Token i cannot see tokens after i.
        # ----------------------------------------------------

        causal_mask = torch.triu(
            torch.ones(
                seq_len,
                seq_len,
                device=input_ids.device,
                dtype=torch.bool
            ),
            diagonal=1
        )

        # ----------------------------------------------------
        # 4. Transformer blocks
        # ----------------------------------------------------

        for block in self.blocks:

            x = block(
                x,
                causal_mask
            )

        # ----------------------------------------------------
        # 5. Vocabulary prediction
        # ----------------------------------------------------

        x = self.final_norm(
            x
        )

        logits = self.output_layer(
            x
        )

        loss = None

        # ----------------------------------------------------
        # 6. Next-token language modelling loss
        # ----------------------------------------------------

        if labels is not None:

            # Example:
            #
            # logits:
            # t0 t1 t2
            #
            # targets:
            # t1 t2 t3

            shift_logits = (
                logits[:, :-1, :]
                .contiguous()
            )

            shift_labels = (
                labels[:, 1:]
                .contiguous()
            )

            loss = F.cross_entropy(
                shift_logits.view(
                    -1,
                    shift_logits.size(-1)
                ),
                shift_labels.view(-1)
            )

        return CausalLMOutput(
            loss=loss,
            logits=logits
        )


# ============================================================
# CREATE OUR CUSTOM MODEL
# ============================================================

model = DecoderTransformerLM(
    vocab_size=len(tokenizer),
    context_length=MODEL_CONTEXT_LENGTH,
    d_model=N_EMBD,
    n_heads=N_HEAD,
    n_layers=N_LAYER,
    d_ff=N_FF,
    dropout=0.1
)


# ============================================================
# MODEL SIZE CHECK
# ============================================================

model_size = sum(
    p.numel()
    for p in model.parameters()
)

print(
    f"Model size: "
    f"{model_size / 1e6:.2f}M parameters"
)

print(
    f"Architecture: "
    f"{N_LAYER} layers, "
    f"{N_HEAD} heads, "
    f"d_model={N_EMBD}, "
    f"head_dim={N_EMBD // N_HEAD}"
)

print(
    "Model class:",
    type(model).__name__
)

# Phase-1 requirement.
assert (
    50_000_000
    <= model_size
    <= 300_000_000
)

# Make absolutely sure we are training
# OUR custom Transformer.
assert isinstance(
    model,
    DecoderTransformerLM
)

Model size: 104.70M parameters
Architecture: 8 layers, 10 heads, d_model=640, head_dim=64
Model class: DecoderTransformerLM


In [11]:

def data_collator(features):
    """
    Stack equal-length token blocks into one batch.

    For causal language modelling:
        input_ids = [t0, t1, t2, ...]
        labels    = [t0, t1, t2, ...]

    The model internally shifts them by one position when computing
    next-token prediction loss.
    """
    input_ids = torch.stack([
        item["input_ids"]
        for item in features
    ])

    return {
        "input_ids": input_ids,
        "labels": input_ids.clone()
    }


## 10. Training Arguments

In [12]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir=MODEL_OUTPUT_DIR,

    num_train_epochs=EPOCHS,

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,

    gradient_accumulation_steps=8,

    learning_rate=3e-4,
    weight_decay=0.1,

    warmup_steps=500,
    lr_scheduler_type="cosine",

    fp16=torch.cuda.is_available(),

    # Evaluate periodically.
    eval_strategy="steps",
    eval_steps=1000,

    # Print training information.
    logging_steps=100,

    # --------------------------------------------------------
    # CHECKPOINTS
    # --------------------------------------------------------

    save_strategy="steps",

    # Save much more frequently than before.
    save_steps=250,

    # Keep latest 4 checkpoints.
    save_total_limit=4,

    # Save optimizer + scheduler as well,
    # so training can actually resume.
    save_only_model=False,

    seed=SEED,

    report_to="none"
)

print("Training arguments created.")
print("Checkpoint interval: 250 optimizer steps")

Training arguments created.
Checkpoint interval: 250 optimizer steps


## 11. Training

In [13]:

from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    processing_class=tokenizer
)

start_time = time.time()

trainer.train()

training_time = (time.time() - start_time)

tokens_per_epoch = (len(train_dataset) * TRAIN_BLOCK_SIZE)
total_tokens = (tokens_per_epoch * EPOCHS)
tokens_per_second = (total_tokens / training_time)

print(f"Training time: {training_time / 3600:.2f} hours")
print(f"Training speed: {tokens_per_second:,.0f} tokens/sec")


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Step,Training Loss,Validation Loss
1000,75.073228,9.310281
2000,64.435210,7.947097
3000,59.168589,7.316020
4000,57.159971,7.091913


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector

Training time: 6.51 hours
Training speed: 6,395 tokens/sec


## 12. Perplexity

In [14]:
eval_results = trainer.evaluate()
eval_loss = eval_results["eval_loss"]
perplexity = math.exp(eval_loss)
print(f"Validation loss: {eval_loss:.6f}")
print(f"Validation perplexity: {perplexity:.4f}")

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Validation loss: 7.069660
Validation perplexity: 1175.7479


In [15]:
import os
import torch

generation_model = DecoderTransformerLM(
    vocab_size=len(tokenizer),
    context_length=MODEL_CONTEXT_LENGTH,
    d_model=560,
    n_heads=10,
    n_layers=8,
    d_ff=2560,
    dropout=0.1
)

safetensor_path = os.path.join(
    MODEL_OUTPUT_DIR,
    "model.safetensors"
)

pytorch_path = os.path.join(
    MODEL_OUTPUT_DIR,
    "pytorch_model.bin"
)

if os.path.exists(safetensor_path):
    from safetensors.torch import load_file
    state_dict = load_file(safetensor_path)
    print("Loading:", safetensor_path)

elif os.path.exists(pytorch_path):
    state_dict = torch.load(
        pytorch_path,
        map_location="cpu",
        weights_only=True
    )
    print("Loading:", pytorch_path)

else:
    raise FileNotFoundError("Saved model weights not found.")

generation_model.load_state_dict(state_dict)

print("✅ Saved custom DecoderTransformerLM loaded successfully.")

FileNotFoundError: Saved model weights not found.

## 13. Plot Training and Validation Loss

In [ ]:
import matplotlib.pyplot as plt

logs = trainer.state.log_history

train_steps = []
train_loss = []
eval_steps = []
eval_loss = []

for log in logs:
    if ("loss" in log and "step" in log):
        train_steps.append(log["step"])
        train_loss.append(log["loss"])
    if ("eval_loss" in log and "step" in log):
        eval_steps.append(log["step"])
        eval_loss.append(log["eval_loss"])

plt.figure(figsize=(10, 5))
plt.plot(train_steps, train_loss, label="Training Loss")
plt.plot(eval_steps, eval_loss, label="Validation Loss")
plt.xlabel("Training Steps")
plt.ylabel("Loss")
plt.title("Phase 1 Training and Validation Loss")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(LOSS_PLOT_PATH, dpi=200)
plt.show()

print("Loss plot saved to:")
print(LOSS_PLOT_PATH)

## 14. Save Final Model to Kaggle Output

In [ ]:

trainer.save_model(MODEL_OUTPUT_DIR)
tokenizer.save_pretrained(MODEL_OUTPUT_DIR)

print("Final model and tokenizer saved to:")
print(MODEL_OUTPUT_DIR)


## 15. Push Model to Hugging Face Hub

In [ ]:
# model.push_to_hub(HF_MODEL_REPO, token=HF_TOKEN)
# tokenizer.push_to_hub(HF_MODEL_REPO, token=HF_TOKEN)

# print("Model and tokenizer pushed successfully.")

## 16. Load Test Data from HuggingFace

In [ ]:
from huggingface_hub import hf_hub_download

In [ ]:
test_file_path = hf_hub_download(
    repo_id=HF_DATASET_REPO,
    filename=TEST_FILE,
    repo_type="dataset",
    token=HF_TOKEN
)

print("Test file:", test_file_path)

## 17. Load Test Prompts

In [ ]:
with open(test_file_path, "r", encoding="utf-8") as f:
    test_data = json.load(f)

print("Number of test examples:", len(test_data))

## 18. Load Final Model for Generation

In [ ]:
from transformers import PreTrainedTokenizerFast
from safetensors.torch import load_file
import torch
import os

# Load OUR saved BPE tokenizer.
generation_tokenizer = (
    PreTrainedTokenizerFast.from_pretrained(
        MODEL_OUTPUT_DIR
    )
)

# Recreate EXACTLY the same custom architecture.
generation_model = DecoderTransformerLM(
    vocab_size=len(generation_tokenizer),
    context_length=MODEL_CONTEXT_LENGTH,
    d_model=N_EMBD,
    n_heads=N_HEAD,
    n_layers=N_LAYER,
    d_ff=N_FF,
    dropout=0.1
)

model_path = os.path.join(
    MODEL_OUTPUT_DIR,
    "model.safetensors"
)

state_dict = load_file(
    model_path
)

generation_model.load_state_dict(
    state_dict
)

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

generation_model = (
    generation_model.to(device)
)

generation_model.eval()

print(
    "Custom DecoderTransformerLM "
    "loaded successfully."
)

print("Device:", device)

print(
    f"Architecture: "
    f"{N_LAYER} layers, "
    f"{N_HEAD} heads, "
    f"d_model={N_EMBD}"
)

print(
    f"Parameters: "
    f"{sum(p.numel() for p in generation_model.parameters()) / 1e6:.2f}M"
)

## 19. Generate 128 Words
#### (words, not tokens)

In [ ]:
def generate_batch(prompts, model, tokenizer, device, target_words=128):
    """
    Generate legal-text continuations using our custom
    DecoderTransformerLM.

    The model predicts one token at a time:
        prompt -> next token -> append -> next token -> ...
    """

    predictions = []

    model.eval()

    for prompt in prompts:

        # --------------------------------------------------------
        # 1. Tokenize the prompt.
        # --------------------------------------------------------
        prompt_ids = tokenizer.encode(
            prompt,
            add_special_tokens=False
        )

        # Keep enough room for generated tokens.
        prompt_ids = prompt_ids[
            -(MODEL_CONTEXT_LENGTH - MAX_GENERATION_TOKENS):
        ]

        generated_ids = []

        # --------------------------------------------------------
        # 2. Autoregressive generation.
        # --------------------------------------------------------
        with torch.inference_mode():

            for _ in range(MAX_GENERATION_TOKENS):

                # Current sequence =
                # original prompt + generated continuation.
                current_ids = prompt_ids + generated_ids

                # Never exceed the model context window.
                current_ids = current_ids[
                    -MODEL_CONTEXT_LENGTH:
                ]

                input_ids = torch.tensor(
                    [current_ids],
                    dtype=torch.long,
                    device=device
                )

                # Forward pass through OUR custom Transformer.
                outputs = model(
                    input_ids=input_ids
                )

                # We only need prediction for the next token.
                next_token_logits = outputs.logits[:, -1, :]

                # ------------------------------------------------
                # Temperature sampling.
                # ------------------------------------------------
                next_token_logits = (
                    next_token_logits / TEMPERATURE
                )

                probabilities = torch.softmax(
                    next_token_logits,
                    dim=-1
                )

                # ------------------------------------------------
                # Top-p / nucleus sampling.
                # ------------------------------------------------
                sorted_probs, sorted_indices = torch.sort(
                    probabilities,
                    descending=True
                )

                cumulative_probs = torch.cumsum(
                    sorted_probs,
                    dim=-1
                )

                # Remove tokens after cumulative probability > TOP_P.
                remove_mask = cumulative_probs > TOP_P

                # Keep at least the most probable token.
                remove_mask[:, 1:] = (
                    remove_mask[:, :-1].clone()
                )
                remove_mask[:, 0] = False

                sorted_probs[remove_mask] = 0.0

                # Re-normalize remaining probabilities.
                sorted_probs = (
                    sorted_probs /
                    sorted_probs.sum(
                        dim=-1,
                        keepdim=True
                    )
                )

                # Sample one token.
                sampled_position = torch.multinomial(
                    sorted_probs,
                    num_samples=1
                )

                next_token = sorted_indices.gather(
                    1,
                    sampled_position
                )

                token_id = int(next_token.item())

                generated_ids.append(token_id)

                # ------------------------------------------------
                # Check how many WORDS we have generated.
                # ------------------------------------------------
                generated_text = tokenizer.decode(
                    generated_ids,
                    skip_special_tokens=True
                )

                if len(generated_text.split()) >= target_words:
                    break

        # --------------------------------------------------------
        # 3. Return exactly the first 128 generated words.
        # --------------------------------------------------------
        generated_text = tokenizer.decode(
            generated_ids,
            skip_special_tokens=True
        )

        generated_words = generated_text.split()
        generated_words = generated_words[:target_words]

        predictions.append(
            " ".join(generated_words)
        )

    return predictions

## 20. Generate Predictions for the Test Set

In [ ]:
predictions = {}
GENERATION_BATCH_SIZE = 32

items = list(test_data.items())
total_examples = len(items)
print(f"Total test examples: {total_examples}")

for start in range(0, total_examples, GENERATION_BATCH_SIZE):
    batch = items[start:start + GENERATION_BATCH_SIZE]
    batch_ids = [str(example_id) for example_id, _ in batch]
    batch_prompts = [record["text"] for _, record in batch]

    batch_predictions = generate_batch(
        prompts=batch_prompts,
        model=generation_model,
        tokenizer=generation_tokenizer,
        device=device,
        target_words=128
    )

    for example_id, prediction in zip(batch_ids, batch_predictions):
        predictions[example_id] = {"text": prediction}

    completed = start + len(batch)
    print(f"Completed {completed}/{total_examples} ({100 * completed / total_examples:.1f}%)")

## 21. Save prediction.json

In [ ]:
with open(PREDICTION_PATH, "w", encoding="utf-8") as f:
    json.dump(
        predictions,
        f,
        ensure_ascii=False,
        indent=2
    )

print("prediction.json saved:")
print(PREDICTION_PATH)

## 22. Create CodaBench Submission ZIP

In [ ]:
with zipfile.ZipFile(SUBMISSION_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as zipf:
    zipf.write(PREDICTION_PATH, arcname="prediction.json")

print("Codabench submission created:")
print(SUBMISSION_ZIP)

In [ ]:
print("CHECKPOINT:")